# Book Genre Classification Using Zero-Shot Learning

This notebook demonstrates how to classify books into simplified genre categories using zero-shot text classification. We'll:

1. **Load and explore** the book dataset
2. **Create simplified categories** by mapping detailed categories to broader genres
3. **Use a pre-trained transformer model** to classify books based on their descriptions
4. **Evaluate model performance** on labeled data
5. **Predict categories** for books with missing category information
6. **Export the enriched dataset** with predicted categories


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from transformers import pipeline
from tqdm import tqdm

# Load the cleaned book dataset
df_books = pd.read_csv("books_cleaned.csv")

print(f"Dataset loaded: {len(df_books)} books")
print(f"Columns: {list(df_books.columns)}")
df_books.head()


Dataset loaded: 5197 books
Columns: ['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count', 'full_title', 'indexed_content']


,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,full_title,indexed_content
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


## 1. Exploring Category Distribution

First, let's examine the distribution of book categories to understand what we're working with.


In [2]:
# Analyze category frequency
category_freq = df_books['categories'].value_counts()
category_stats = category_freq.reset_index()
category_stats.columns = ['category', 'book_count']
category_stats = category_stats.sort_values('book_count', ascending=False)

print(f"Total unique categories: {len(category_stats)}")
print("\nTop 15 categories:")
print(category_stats.head(15))


Total unique categories: 479

Top 15 categories:
                     category  book_count
0                     Fiction        2111
1            Juvenile Fiction         390
2   Biography & Autobiography         311
3                     History         207
4          Literary Criticism         124
5                  Philosophy         117
6                    Religion         117
7     Comics & Graphic Novels         116
8                       Drama          86
9         Juvenile Nonfiction          57
10                    Science          56
11                     Poetry          51
12       Literary Collections          50
13       Business & Economics          49
14             Social Science          48


In [3]:
# Filter to categories with substantial representation (more than 50 books)
major_categories = category_stats[category_stats['book_count'] > 50]

print(f"Categories with more than 50 books: {len(major_categories)}")
print("\nMajor categories:")
print(major_categories)


Categories with more than 50 books: 12

Major categories:
                     category  book_count
0                     Fiction        2111
1            Juvenile Fiction         390
2   Biography & Autobiography         311
3                     History         207
4          Literary Criticism         124
5                  Philosophy         117
6                    Religion         117
7     Comics & Graphic Novels         116
8                       Drama          86
9         Juvenile Nonfiction          57
10                    Science          56
11                     Poetry          51


## 2. Creating Simplified Category Mapping

We'll consolidate the many specific categories into broader, more manageable genre classifications. This simplifies the classification task and makes the results more interpretable.


In [4]:
# Define mapping from specific categories to simplified genres
genre_mapping = {
    'Fiction': "Fiction",
    'Juvenile Fiction': "Children's Fiction",
    'Biography & Autobiography': "Nonfiction",
    'History': "Nonfiction",
    'Literary Criticism': "Nonfiction",
    'Philosophy': "Nonfiction",
    'Religion': "Nonfiction",
    'Comics & Graphic Novels': "Fiction",
    'Drama': "Fiction",
    'Juvenile Nonfiction': "Children's Nonfiction",
    'Science': "Nonfiction",
    'Poetry': "Fiction"
}

# Apply the mapping to create simplified genre column
df_books['genre'] = df_books['categories'].map(genre_mapping)

# Check how many books got mapped
mapped_count = df_books['genre'].notna().sum()
print(f"Books successfully mapped to genres: {mapped_count} out of {len(df_books)}")
print(f"Mapping coverage: {mapped_count/len(df_books)*100:.2f}%")


Books successfully mapped to genres: 3743 out of 5197
Mapping coverage: 72.02%


In [5]:
# Display books with successfully mapped genres
books_with_genres = df_books[df_books['genre'].notna()].copy()

print(f"Books with genre labels: {len(books_with_genres)}")
print(f"\nGenre distribution:")
print(books_with_genres['genre'].value_counts())


Books with genre labels: 3743

Genre distribution:
genre
Fiction                  2364
Nonfiction                932
Children's Fiction        390
Children's Nonfiction      57
Name: count, dtype: int64


## 3. Setting Up Zero-Shot Classification Model

We'll use a pre-trained transformer model (BART-large-MNLI) for zero-shot classification. This model can classify text into categories it hasn't been explicitly trained on, making it perfect for our use case.


In [6]:
# Initialize the zero-shot classification pipeline
# Using BART-large-MNLI model which is excellent for zero-shot tasks
classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device="mps"  # Use Metal Performance Shaders on Mac
)

# Define the target genre classes for classification
target_genres = ["Fiction", "Nonfiction"]

print("Zero-shot classifier initialized successfully")


Zero-shot classifier initialized successfully


## 4. Testing the Classifier

Let's test the classifier on a sample book description to see how it works.


In [7]:
# Get a sample Fiction book description for testing
fiction_books = books_with_genres[books_with_genres['genre'] == 'Fiction']
sample_text = fiction_books['description'].iloc[0]

print("Sample book description (first 200 characters):")
print(sample_text[:200] + "...")
print("\n" + "="*50)

# Classify the sample
result = classifier(sample_text, target_genres)
print(f"\nClassification result:")
print(f"Predicted genre: {result['labels'][0]}")
print(f"Confidence scores: {dict(zip(result['labels'], result['scores']))}")


Sample book description (first 200 characters):
A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the...


Classification result:
Predicted genre: Fiction
Confidence scores: {'Fiction': 0.843826413154602, 'Nonfiction': 0.15617360174655914}


## 5. Creating a Classification Function

We'll create a helper function that extracts the predicted label with the highest confidence score.


In [8]:
def classify_book_genre(text, genre_options):
    """
    Classify a book description into one of the provided genre categories.
    
    Args:
        text: Book description text
        genre_options: List of possible genre labels
        
    Returns:
        Predicted genre label (string)
    """
    classification_result = classifier(text, genre_options)
    # Get the index of the highest scoring label
    best_match_idx = np.argmax(classification_result['scores'])
    predicted_genre = classification_result['labels'][best_match_idx]
    return predicted_genre

# Test the function
test_prediction = classify_book_genre(sample_text, target_genres)
print(f"Function test - Predicted genre: {test_prediction}")


Function test - Predicted genre: Fiction


## 6. Evaluating Model Performance

Before using the model to predict missing categories, let's evaluate its performance on books where we already know the genre. We'll test on 300 Fiction and 300 Nonfiction books.


In [9]:
# Prepare evaluation data
true_labels = []
predicted_labels = []

# Classify Fiction books
fiction_descriptions = fiction_books['description'].reset_index(drop=True)
num_fiction_samples = min(300, len(fiction_descriptions))

print(f"Classifying {num_fiction_samples} Fiction books...")
for idx in tqdm(range(num_fiction_samples), desc="Fiction books"):
    book_description = fiction_descriptions[idx]
    predicted = classify_book_genre(book_description, target_genres)
    predicted_labels.append(predicted)
    true_labels.append("Fiction")


Classifying 300 Fiction books...


Fiction books: 100%|██████████| 300/300 [00:47<00:00,  6.29it/s]


In [10]:
# Classify Nonfiction books
nonfiction_books = books_with_genres[books_with_genres['genre'] == 'Nonfiction']
nonfiction_descriptions = nonfiction_books['description'].reset_index(drop=True)
num_nonfiction_samples = min(300, len(nonfiction_descriptions))

print(f"Classifying {num_nonfiction_samples} Nonfiction books...")
for idx in tqdm(range(num_nonfiction_samples), desc="Nonfiction books"):
    book_description = nonfiction_descriptions[idx]
    predicted = classify_book_genre(book_description, target_genres)
    predicted_labels.append(predicted)
    true_labels.append("Nonfiction")


Classifying 300 Nonfiction books...


Nonfiction books: 100%|██████████| 300/300 [00:42<00:00,  7.00it/s]


In [11]:
# Create evaluation results dataframe
evaluation_results = pd.DataFrame({
    'true_genre': true_labels,
    'predicted_genre': predicted_labels
})

# Calculate accuracy
evaluation_results['is_correct'] = (
    evaluation_results['true_genre'] == evaluation_results['predicted_genre']
)
accuracy = evaluation_results['is_correct'].mean()

print(f"\nModel Performance:")
print(f"Total samples evaluated: {len(evaluation_results)}")
print(f"Correct predictions: {evaluation_results['is_correct'].sum()}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Show confusion matrix summary
print("\nPrediction breakdown:")
print(evaluation_results.groupby(['true_genre', 'predicted_genre']).size().unstack(fill_value=0))



Model Performance:
Total samples evaluated: 600
Correct predictions: 467
Accuracy: 0.7783 (77.83%)

Prediction breakdown:
predicted_genre  Fiction  Nonfiction
true_genre                          
Fiction              204          96
Nonfiction            37         263


## 7. Predicting Categories for Books with Missing Genres

Now we'll use the classifier to predict genres for books that don't have a mapped category. This will help us complete the dataset.


In [12]:
# Identify books without genre labels
books_missing_genre = df_books[df_books['genre'].isna()].copy()
books_missing_genre = books_missing_genre[['isbn13', 'description']].reset_index(drop=True)

print(f"Books needing genre prediction: {len(books_missing_genre)}")


Books needing genre prediction: 1454


In [13]:
# Predict genres for all books with missing categories
isbn_list = []
predicted_genre_list = []

print("Predicting genres for books with missing categories...")
for idx in tqdm(range(len(books_missing_genre)), desc="Processing books"):
    book_isbn = books_missing_genre['isbn13'].iloc[idx]
    book_description = books_missing_genre['description'].iloc[idx]
    
    predicted = classify_book_genre(book_description, target_genres)
    
    isbn_list.append(book_isbn)
    predicted_genre_list.append(predicted)


Predicting genres for books with missing categories...


Processing books: 100%|██████████| 1454/1454 [02:35<00:00,  9.37it/s]


In [14]:
# Create dataframe with predictions
genre_predictions = pd.DataFrame({
    'isbn13': isbn_list,
    'predicted_genre': predicted_genre_list
})

print(f"Predictions generated for {len(genre_predictions)} books")
print("\nPredicted genre distribution:")
print(genre_predictions['predicted_genre'].value_counts())


Predictions generated for 1454 books

Predicted genre distribution:
predicted_genre
Nonfiction    1010
Fiction        444
Name: count, dtype: int64


## 8. Merging Predictions with Original Dataset

We'll merge the predicted genres back into the main dataset, filling in the missing values.


In [15]:
# Merge predictions with original dataset
df_books = df_books.merge(genre_predictions, on='isbn13', how='left')

# Fill missing genres with predictions
df_books['genre'] = df_books['genre'].fillna(df_books['predicted_genre'])

# Drop the temporary prediction column
df_books = df_books.drop(columns=['predicted_genre'])

print(f"Final dataset size: {len(df_books)}")
print(f"Books with genre labels: {df_books['genre'].notna().sum()}")
print(f"\nFinal genre distribution:")
print(df_books['genre'].value_counts())


Final dataset size: 5197
Books with genre labels: 5197

Final genre distribution:
genre
Fiction                  2808
Nonfiction               1942
Children's Fiction        390
Children's Nonfiction      57
Name: count, dtype: int64


## 9. Exporting the Enriched Dataset

Finally, we'll save the dataset with all genre classifications to a CSV file for use in downstream tasks.


In [16]:
# Export the enriched dataset
output_file = 'books_with_categories.csv'
df_books.to_csv(output_file, index=False)

print(f"Dataset exported to: {output_file}")
print(f"Total books: {len(df_books)}")
print(f"Books with genre: {df_books['genre'].notna().sum()}")
print(f"\nDataset columns: {list(df_books.columns)}")


Dataset exported to: books_with_categories.csv
Total books: 5197
Books with genre: 5197

Dataset columns: ['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count', 'full_title', 'indexed_content', 'genre']
